# BTC Indicator Check

A simple Bitcoin indicator scanner using live market data and technical indicators. This tool displays signals only — it does not execute any trades.

---

## How to Use
1. Run **all cells top to bottom** (Cell → Run All)
2. The last two cells will output live indicators and signals
3. To refresh, just re-run the last two cells — no need to restart

---

## What It Does

| Cell | Description |
|---|---|
| Install cells | Sets up `anthropic`, `requests`, and `ta` (technical analysis library) |
| Coinbase price fetcher | Pulls live BTC spot/buy/sell price from Coinbase API |
| Indicators cell | Fetches 30 days of candle data from CoinGecko, calculates RSI, MACD, Bollinger Bands, EMA, SMA |
| Signals cell | Fetches 1500 days of daily candles from Coinbase for 200W SMA, scores RSI/MACD/BBands, prints verdict |

---

## Indicators Used

- **RSI (14)** — Momentum. Below 30 = oversold (buy), above 70 = overbought (sell)
- **MACD (12/26/9)** — Trend. Bullish/bearish crossover signals
- **Bollinger Bands (20)** — Volatility. Price at lower band = buy, upper band = sell
- **200-Week SMA** — Macro floor. Historically a major buy zone when price is below it

In [ ]:
pip install anthropic

In [ ]:
import os
import anthropic

client = anthropic.Anthropic(api_key=os.environ.get("ANTHROPIC_API_KEY"))

def analyze_with_claude(signals, current_price, sma_200w, below_200w, rsi_value):
    """Send BTC indicator data to Claude and return a brief technical analysis."""
    signals_text = "\n".join(f"- {d}: {r}" for d, r in signals)
    sma_gap = f"${sma_200w - current_price:,.2f} below" if below_200w else f"${current_price - sma_200w:,.2f} above"
    prompt = (
        f"You are a concise technical analysis assistant. "
        f"Here are the current BTC market indicators:\n\n"
        f"Current Price: ${current_price:,.2f}\n"
        f"200-Week SMA: ${sma_200w:,.2f} (price is {sma_gap} the 200W SMA)\n"
        f"RSI (14): {rsi_value:.1f}\n\n"
        f"Indicator signals:\n{signals_text}\n\n"
        f"Give a 2-3 sentence technical summary of what these signals suggest. "
        f"Do not give financial advice."
    )
    message = client.messages.create(
        model="claude-sonnet-4-6",
        max_tokens=256,
        messages=[{"role": "user", "content": prompt}]
    )
    return message.content[0].text

print("Anthropic client initialized and analyze_with_claude() ready.")


In [ ]:
pip install requests

In [ ]:
import requests
from datetime import datetime

def get_bitcoin_price(symbol: str = "BTC-USD") -> dict:
    BASE_URL = "https://api.coinbase.com/v2"
    try:
        response = requests.get(
            f"{BASE_URL}/prices/{symbol}/spot",
            timeout=10
        )
        response.raise_for_status()
        data = response.json()
        return {
            "symbol": symbol,
            "price": float(data["data"]["amount"]),
            "currency": data["data"]["currency"],
            "timestamp": datetime.now().isoformat(),
            "source": "Coinbase"
        }
    except requests.exceptions.RequestException as e:
        raise RuntimeError(f"An unexpected error occurred: {e}")

def get_bitcoin_detailed_stats(symbol: str = "BTC-USD") -> dict:
    BASE_URL = "https://api.coinbase.com/v2"
    try:
        buy = requests.get(f"{BASE_URL}/prices/{symbol}/buy", timeout=10).json()
        sell = requests.get(f"{BASE_URL}/prices/{symbol}/sell", timeout=10).json()
        spot = requests.get(f"{BASE_URL}/prices/{symbol}/spot", timeout=10).json()
        return {
            "symbol": symbol,
            "spot_price": float(spot["data"]["amount"]),
            "buy_price": float(buy["data"]["amount"]),
            "sell_price": float(sell["data"]["amount"]),
            "timestamp": datetime.now().isoformat(),
            "source": "Coinbase"
        }
    except requests.exceptions.RequestException as e:
        raise RuntimeError(f"Failed to fetch data: {e}")

# Run it
print("=== Current Bitcoin Price ===")
price_data = get_bitcoin_price()
print(f"Symbol:    {price_data['symbol']}")
print(f"Price:     ${price_data['price']:,.2f} {price_data['currency']}")
print(f"Timestamp: {price_data['timestamp']}")

print("\n=== Bitcoin Buy/Sell Stats ===")
stats = get_bitcoin_detailed_stats()
print(f"Spot Price: ${stats['spot_price']:,.2f}")
print(f"Buy Price:  ${stats['buy_price']:,.2f}")
print(f"Sell Price: ${stats['sell_price']:,.2f}")

In [ ]:
pip install ta

In [ ]:
import requests
import pandas as pd
from datetime import datetime
from ta.momentum import RSIIndicator
from ta.trend import MACD, SMAIndicator, EMAIndicator
from ta.volatility import BollingerBands

def get_btc_candles(days=30):
    """Fetch BTC OHLCV candle data from CoinGecko (public, no auth needed)."""
    response = requests.get(
        "https://api.coingecko.com/api/v3/coins/bitcoin/ohlc",
        params={"vs_currency": "usd", "days": days},
        timeout=10
    )
    response.raise_for_status()
    data = response.json()
    df = pd.DataFrame(data, columns=['timestamp', 'open', 'high', 'low', 'close'])
    df['timestamp'] = pd.to_datetime(df['timestamp'], unit='ms')
    df = df.sort_values('timestamp').reset_index(drop=True)
    return df

# Fetch 30 days of candles
df = get_btc_candles(days=30)

# --- RSI (14) ---
df['RSI'] = RSIIndicator(df['close'], window=14).rsi()

# --- MACD (12, 26, 9) ---
macd = MACD(df['close'])
df['MACD_12_26_9'] = macd.macd()
df['MACDs_12_26_9'] = macd.macd_signal()
df['MACDh_12_26_9'] = macd.macd_diff()

# --- Bollinger Bands (20, 2) ---
bb = BollingerBands(df['close'], window=20, window_dev=2)
df['BBU_20_2.0'] = bb.bollinger_hband()
df['BBM_20_2.0'] = bb.bollinger_mavg()
df['BBL_20_2.0'] = bb.bollinger_lband()

# --- EMA & SMA ---
df['EMA_20'] = EMAIndicator(df['close'], window=20).ema_indicator()
df['SMA_50'] = SMAIndicator(df['close'], window=50).sma_indicator()

# --- Print latest values ---
latest = df.iloc[-1]
print("=== Latest BTC Indicators ===")
print(f"Timestamp:    {latest['timestamp']}")
print(f"Close Price:  ${latest['close']:,.2f}")

print(f"\n--- RSI (14) ---")
print(f"RSI:          {latest['RSI']:.2f}  {'(Overbought >70)' if latest['RSI'] > 70 else '(Oversold <30)' if latest['RSI'] < 30 else '(Neutral)'}")

print(f"\n--- MACD (12/26/9) ---")
print(f"MACD:         {latest['MACD_12_26_9']:.2f}")
print(f"Signal:       {latest['MACDs_12_26_9']:.2f}")
print(f"Histogram:    {latest['MACDh_12_26_9']:.2f}")

print(f"\n--- Bollinger Bands (20) ---")
print(f"Upper Band:   ${latest['BBU_20_2.0']:,.2f}")
print(f"Middle Band:  ${latest['BBM_20_2.0']:,.2f}")
print(f"Lower Band:   ${latest['BBL_20_2.0']:,.2f}")

print(f"\n--- Moving Averages ---")
print(f"EMA (20):     ${latest['EMA_20']:,.2f}")
print(f"SMA (50):     ${latest['SMA_50']:,.2f}")

print("\n=== Last 5 Rows ===")
cols = ['timestamp', 'close', 'RSI', 'MACD_12_26_9', 'EMA_20', 'SMA_50']
print(df[cols].tail().to_string(index=False))

In [ ]:
from datetime import datetime, timedelta, timezone
from ta.trend import SMAIndicator

# ── 1. Fetch daily BTC candles from Coinbase (batched) ─────────────────────
def get_btc_daily_history(total_days=1500):
    """Fetch daily BTC candles from Coinbase, batched in 300-day chunks."""
    all_candles = []
    end = datetime.now(timezone.utc)

    while total_days > 0:
        batch = min(total_days, 300)
        start = end - timedelta(days=batch)
        response = requests.get(
            "https://api.exchange.coinbase.com/products/BTC-USD/candles",
            params={
                "granularity": 86400,
                "start": start.isoformat(),
                "end": end.isoformat()
            },
            timeout=15
        )
        response.raise_for_status()
        data = response.json()
        all_candles.extend(data)
        end = start
        total_days -= batch

    # Columns: [time, low, high, open, close, volume]
    df_raw = pd.DataFrame(all_candles, columns=['timestamp', 'low', 'high', 'open', 'close', 'volume'])
    df_raw['timestamp'] = pd.to_datetime(df_raw['timestamp'], unit='s')
    df_raw = df_raw.drop_duplicates(subset='timestamp').sort_values('timestamp').reset_index(drop=True)
    return df_raw

# ── 2. Resample to weekly and calculate 200W SMA ──────────────────────────
daily_df = get_btc_daily_history(total_days=1500)
weekly_df = daily_df.set_index('timestamp').resample('W')['close'].last().reset_index()
weekly_df['SMA_200W'] = SMAIndicator(weekly_df['close'], window=200).sma_indicator()

current_200w_sma = weekly_df['SMA_200W'].iloc[-1]
current_price    = df['close'].iloc[-1]
latest           = df.iloc[-1]

# ── 3. Signal logic ───────────────────────────────────────────────────────
signals = []

# RSI
if latest['RSI'] < 30:
    signals.append(("BUY",  "RSI oversold (< 30)"))
elif latest['RSI'] > 70:
    signals.append(("SELL", "RSI overbought (> 70)"))
else:
    signals.append(("NEUTRAL", f"RSI neutral ({latest['RSI']:.1f})"))

# MACD crossover
if latest['MACDh_12_26_9'] > 0 and df['MACDh_12_26_9'].iloc[-2] <= 0:
    signals.append(("BUY",  "MACD bullish crossover"))
elif latest['MACDh_12_26_9'] < 0 and df['MACDh_12_26_9'].iloc[-2] >= 0:
    signals.append(("SELL", "MACD bearish crossover"))
else:
    signals.append(("NEUTRAL", "MACD no crossover"))

# Bollinger Bands
if latest['close'] <= latest['BBL_20_2.0']:
    signals.append(("BUY",  "Price at/below lower Bollinger Band"))
elif latest['close'] >= latest['BBU_20_2.0']:
    signals.append(("SELL", "Price at/above upper Bollinger Band"))
else:
    signals.append(("NEUTRAL", "Price inside Bollinger Bands"))

# ── 4. Tally ─────────────────────────────────────────────────────────────
buy_count  = sum(1 for d, _ in signals if d == "BUY")
sell_count = sum(1 for d, _ in signals if d == "SELL")
below_200w = current_price < current_200w_sma

# ── 5. Output ─────────────────────────────────────────────────────────────
W = 58
print("=" * W)
print(f"  BTC TRADING SIGNALS  |  {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("=" * W)
print(f"  Current Price :  ${current_price:>12,.2f}")
print(f"  200W SMA      :  ${current_200w_sma:>12,.2f}")

if below_200w:
    gap = current_200w_sma - current_price
    print()
    print("  " + "*" * (W - 4))
    print(f"  ***  ALERT: BTC IS BELOW THE 200-WEEK SMA!  ***")
    print(f"  ***  ${gap:,.2f} below — historically extreme low zone  ***")
    print("  " + "*" * (W - 4))
else:
    gap = current_price - current_200w_sma
    print(f"\n  [OK] Price is ${gap:,.2f} ABOVE the 200W SMA")

print("\n" + "-" * W)
print("  INDICATOR SIGNALS:")
print("-" * W)

for direction, reason in signals:
    if direction == "BUY":
        tag = "[ BUY  ]"
    elif direction == "SELL":
        tag = "[ SELL ]"
    else:
        tag = "[NEUTRAL]"
    print(f"  {tag}  {reason}")

print("-" * W)
if buy_count > sell_count:
    verdict = f"BULLISH  ({buy_count} buy vs {sell_count} sell signals)"
elif sell_count > buy_count:
    verdict = f"BEARISH  ({sell_count} sell vs {buy_count} buy signals)"
else:
    verdict = f"NEUTRAL  ({buy_count} buy vs {sell_count} sell signals)"

print(f"  OVERALL: {verdict}")
print("=" * W)
# ── 6. Claude AI Analysis ───────────────────────────────────────────
import textwrap
print()
print("-" * W)
print("  AI ANALYSIS (Claude):")
print("-" * W)
analysis = analyze_with_claude(signals, current_price, current_200w_sma, below_200w, latest['RSI'])
for line in textwrap.wrap(analysis, width=W - 4):
    print(f"  {line}")
print("=" * W)
